<a href="https://colab.research.google.com/github/effat/MLP-Demo/blob/main/NLP_Demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

We will build a machine learning model that can tell whether a sentence expresses a positive or negative emotion.
We will use an LSTM (Long Short-Term Memory) network for this task.

We will compare two ways of representing words:


1.   Random embeddings (the model learns word meanings from scratch as vector)
2.   Pretrained GloVe embeddings (vector embeeding from a huge text collection)



# **Import Libraries**

In [1]:
import numpy as np
import pandas as pd
import re
import string
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split


# **Dataset**

In [2]:
data = {
    'text': [
        "I absolutely loved this movie! It was fantastic.",
        "What a wonderful story, I’ll watch it again.",
        "Wow! We have won the match",
        "Disgusting! The player destroyed the close to winning match",
        "This was the worst film I’ve seen in my life.",
        "What a gloomy weather today! I hate to go to college.",
        "The food and the ambience of the restaurant was amazing.",
        "Not good, I wouldn’t recommend it.",
        "Her wedding dress was beautiful and she looked georgeous.",
        "Sitting in the hall waiting for the movie to end is just too long and quite dull."
    ],
    'label': ['positive', 'positive', 'positive', 'negative', 'negative',
              'negative', 'positive', 'negative', 'positive', 'negative']
}

df = pd.DataFrame(data)
df.head()

,text,label
0,I absolutely loved this movie! It was fantastic.,positive
1,"What a wonderful story, I’ll watch it again.",positive
2,Wow! We have won the match,positive
3,Disgusting! The player destroyed the close to ...,negative
4,This was the worst film I’ve seen in my life.,negative


**Data Cleaning and Preprocessing**

Text in its original form (like movie reviews) contains punctuation, capitalization, and extra words that do not contribute much for prediction.
Preprocessing cleans the text to make it easier for the computer to understand.

**We will:**

1.   Turn all the text into lowercase
2.   Remove punctuation and numbers
3.   Remove stopwords (common words like “the”, “is”, “at” that don’t add meaning. Articles, auxiliary verbs, prepositions are example of stop words. Stop word listings for English language are available in Python.)





In [3]:
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+", "", text)          # remove URLs
    text = re.sub(r"[^a-z\s]", "", text)         # keep only letters
    text = text.translate(str.maketrans("", "", string.punctuation))
    tokens = text.split()
    tokens = [word for word in tokens if word not in stop_words]
    return " ".join(tokens)

df['clean_text'] = df['text'].apply(clean_text)
df[['text', 'clean_text']].head()


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


,text,clean_text
0,I absolutely loved this movie! It was fantastic.,absolutely loved movie fantastic
1,"What a wonderful story, I’ll watch it again.",wonderful story ill watch
2,Wow! We have won the match,wow match
3,Disgusting! The player destroyed the close to ...,disgusting player destroyed close winning match
4,This was the worst film I’ve seen in my life.,worst film ive seen life


**Label Encoding (One-Hot)**

The computer does not understand text labels like “positive” or “negative.”
So we convert them into numbers: a process called encoding.
As we have two class labels [positive, negative], we will use a vector of length 2 to map class labels.

We will use one-hot encoding. In one-hot encoding, all values are zero except for one position that is set to one for that class label. In our case,

positive → [1, 0]

negative → [0, 1]

In [4]:
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical

label_encoder = LabelEncoder()
integer_labels = label_encoder.fit_transform(df['label'])
y = to_categorical(integer_labels)


**Tokenization**

In our example, our inputs are sentence. Each sentence have different length.

Tokenization is the process of splitting sentences into smaller parts, usually words.

Each word is then assigned a unique number (an “index”) that represents it in **our vocabulary**.

For example:

“I love this movie” → [1, 23, 5, 10]

Here, the sentence is converted to a list of numbers denoting the index of the word in our vocabulary.

Then we make all lists the same length by adding zeros (padding) at the end.

In [5]:
max_words = 10000   # vocabulary size limit
max_len = 20        # all reviews will be padded/truncated to 20 words

tokenizer = Tokenizer(num_words=max_words, oov_token="<OOV>")
tokenizer.fit_on_texts(df['clean_text'])
sequences = tokenizer.texts_to_sequences(df['clean_text'])
word_index = tokenizer.word_index

X = pad_sequences(sequences, maxlen=max_len, padding='post')


**Split data for training and testing**

In [6]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)


**Understanding Word Embeddings**

Before text can be fed into an LSTM, words must be represented as vectors (lists of numbers).

These vectors capture meaning, for example, the words “happy” and “joyful” will have similar representations.

We will compare two types:


1. **Random Embedding**	The model starts with random numbers and learns word meanings during training from training examples.

2. **Pretrained Embedding**	Uses word vectors already trained on billions of words on a large English corpus. We will use Golve embeddings.

**LSTM with Random Embedding**

In [8]:
embedding_dim = 50

model_random = Sequential([
    Embedding(input_dim=max_words, output_dim=embedding_dim, input_length=max_len),
    LSTM(128, dropout=0.2, recurrent_dropout=0.2),
    Dense(2, activation='softmax')
])

model_random.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model_random.summary()

history_random = model_random.fit(X_train, y_train, epochs=10, batch_size=8, validation_split=0.2)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 5s 5s/step - accuracy: 0.2000 - loss: 0.6943 - val_accuracy: 0.5000 - val_loss: 0.6937
Epoch 2/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 126ms/step - accuracy: 0.6000 - loss: 0.6849 - val_accuracy: 0.5000 - val_loss: 0.6948
Epoch 3/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 119ms/step - accuracy: 0.6000 - loss: 0.6844 - val_accuracy: 0.5000 - val_loss: 0.6964
Epoch 4/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 123ms/step - accuracy: 0.6000 - loss: 0.6881 - val_accuracy: 0.5000 - val_loss: 0.6981
Epoch 5/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 136ms/step - accuracy: 0.6000 - loss: 0.6699 - val_accuracy: 0.5000 - val_loss: 0.7004
Epoch 6/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 122ms/step - accuracy: 0.6000 - loss: 0.6828 - val_accuracy: 0.5000 - val_loss: 0.7027
Epoch 7/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 121ms/step - accuracy: 0.6000 - loss: 0.6742 - val_accuracy: 0.5000 - val_loss: 0.7055
Epoch 8/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 125ms/step - accuracy: 0.6000 - loss: 0.6773 - val_accuracy: 0.5000 - val_loss: 0.

**LSTM with Pretrained GloVe Embedding**
We need to install gensim.

In [10]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 79.1 MB/s eta 0:00:00


In [11]:
import gensim.downloader as api

# This downloads a 50-dimensional GloVe model trained on Twitter data
wv = api.load('glove-twitter-50')
embedding_dim = wv.vector_size
print("Loaded GloVe model with dimension:", embedding_dim)


[==================================================] 100.0% 199.5/199.5MB downloaded
Loaded GloVe model with dimension: 50


Now we will create an embedding matrix that aligns our tokenizer’s word indices with the GloVe vectors.

In [12]:
embedding_matrix = np.zeros((max_words, embedding_dim))
for word, i in word_index.items():
    if i < max_words:
        if word in wv:
            embedding_matrix[i] = wv[word]


**LSTM Using Pretrained Embedding**

In [13]:
model_glove = Sequential([
    Embedding(input_dim=max_words, output_dim=embedding_dim,
              weights=[embedding_matrix], input_length=max_len, trainable=False),
    LSTM(128, dropout=0.2, recurrent_dropout=0.2),
    Dense(2, activation='softmax')
])

model_glove.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model_glove.summary()
history_glove = model_glove.fit(X_train, y_train, epochs=10, batch_size=8, validation_split=0.2)


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ ?                      │       500,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 500,000 (1.91 MB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 500,000 (1.91 MB)

Epoch 1/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 8s 8s/step - accuracy: 0.6000 - loss: 0.6824 - val_accuracy: 0.5000 - val_loss: 0.6839
Epoch 2/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 117ms/step - accuracy: 0.6000 - loss: 0.6726 - val_accuracy: 0.5000 - val_loss: 0.6801
Epoch 3/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 126ms/step - accuracy: 0.6000 - loss: 0.6634 - val_accuracy: 0.5000 - val_loss: 0.6766
Epoch 4/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 128ms/step - accuracy: 0.6000 - loss: 0.6149 - val_accuracy: 0.5000 - val_loss: 0.6740
Epoch 5/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 112ms/step - accuracy: 0.6000 - loss: 0.6189 - val_accuracy: 0.5000 - val_loss: 0.6749
Epoch 6/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 113ms/step - accuracy: 0.6000 - loss: 0.6574 - val_accuracy: 0.5000 - val_loss: 0.6795
Epoch 7/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 141ms/step - accuracy: 0.6000 - loss: 0.5757 - val_accuracy: 0.5000 - val_loss: 0.6875
Epoch 8/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 114ms/step - accuracy: 0.6000 - loss: 0.5413 - val_accuracy: 0.5000 - val_loss: 0.